In [ ]:
import pandas as pd

# Load the uploaded CSV file into a pandas DataFrame
df = pd.read_csv('/content/Snitch_Fashion_Sales_Uncleaned.csv')

# Display the first 5 rows of the DataFrame to confirm it's loaded correctly
display(df.head())

,Order_ID,Customer_Name,Product_Category,Product_Name,Units_Sold,Unit_Price,Discount_%,Sales_Amount,Order_Date,City,Segment,Profit
0,1000,Brian Thompson,Jeans,Slim Fit Jeans,NaN,842.00,0.60,0.00,2025-02-27,Delhi,B2C,2137.45
1,1001,Shaun Ross,Jeans,Slim Fit Jeans,1.0,NaN,NaN,0.00,2025-07-15,Ahmedabad,NaN,1588.15
2,1002,Sarah Snyder,Jackets,Puffer Coat,1.0,637.82,NaN,0.00,02-01-2025,Mumbai,B2B,-158.03
3,1003,Jay Briggs,Shoes,Loafers,2.0,2962.27,NaN,0.00,18-06-2025,bengaluru,B2B,2296.50
4,1004,Maria Blake,Accessories,Belts,1.0,2881.07,0.27,2103.18,NaN,hyderbad,NaN,63.66


In [ ]:
df.isnull().sum()


,0
Order_ID,0
Customer_Name,0
Product_Category,0
Product_Name,0
Units_Sold,1306
Unit_Price,1210
Discount_%,1651
Sales_Amount,0
Order_Date,606
City,0


In [26]:
df.describe()

,Order_ID,Units_Sold,Unit_Price,Discount_%,Sales_Amount,Order_Date,Profit
count,2391.000000,2391.000000,2391.000000,2391.000000,2391.000000,599,2391.000000
mean,2246.314931,1.967378,2690.329529,0.649778,125.786253,2024-07-17 17:25:44.574290432,975.371196
min,1000.000000,-2.000000,400.210000,0.000000,-7518.330000,2023-07-23 00:00:00,-992.610000
25%,1628.500000,2.000000,2637.585000,0.650000,0.000000,2024-01-19 12:00:00,-18.710000
50%,2239.000000,2.000000,2685.985000,0.650000,0.000000,2024-07-26 00:00:00,938.710000
75%,2869.500000,2.000000,2718.915000,0.650000,0.000000,2025-01-22 12:00:00,1989.060000
max,3499.000000,6.000000,4998.910000,1.300000,29180.680000,2025-07-22 00:00:00,2996.490000
std,721.135170,1.812261,949.121211,0.221724,1476.421421,NaN,1155.028706


In [27]:
# 1. Fix negative Units_Sold
df.loc[df['Units_Sold'] < 0, 'Units_Sold'] = df['Units_Sold'].median()


In [28]:
# 2. Fix negative Sales_Amount
df.loc[df['Sales_Amount'] < 0, 'Sales_Amount'] = df['Sales_Amount'].median()


In [29]:
# 3. Fix invalid Discount_% above 1
df.loc[df['Discount_%'] > 1, 'Discount_%'] = df['Discount_%'].median()


In [30]:
df['Units_Sold'] = df['Units_Sold'].fillna(df['Units_Sold'].median())

In [31]:
df['Unit_Price'] = df['Unit_Price'].fillna(df['Unit_Price'].median())

In [32]:
df['Discount_%'] = df['Discount_%'].fillna(df['Discount_%'].median())

In [33]:
df['Segment'] = df['Segment'].fillna(df['Segment'].mode()[0])

In [34]:
# 5. Fix Order_Date
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Order_Date'] = df['Order_Date'].fillna(df['Order_Date'].mode()[0])


In [35]:
df.isnull().sum()

,0
Order_ID,0
Customer_Name,0
Product_Category,0
Product_Name,0
Units_Sold,0
Unit_Price,0
Discount_%,0
Sales_Amount,0
Order_Date,0
City,0


Negative values in Units_Sold and Sales_Amount were treated as invalid entries because sales quantity and sales amount cannot be negative. These values were replaced using median imputation.

Discount values greater than 1 were considered invalid because the discount column was stored in decimal form, where 1 represents 100%. Values above 1 were replaced using the median discount.

In [38]:
df.duplicated().sum()

np.int64(0)

In [39]:
df['Order_ID'].duplicated().sum()

np.int64(0)

In [40]:
# Check invalid values
print("Negative Units Sold:", (df['Units_Sold'] < 0).sum())
print("Negative Sales Amount:", (df['Sales_Amount'] < 0).sum())
print("Discount above 100%:", (df['Discount_%'] > 1).sum())
print("Invalid Dates:", pd.to_datetime(df['Order_Date'], errors='coerce').isnull().sum())

Negative Units Sold: 0
Negative Sales Amount: 0
Discount above 100%: 0
Invalid Dates: 0


In [41]:
df.to_csv("cleaned_snitch_fashion_sales.csv", index=False)

After cleaning, the dataset contains zero missing values, zero duplicate rows, zero duplicate Order_ID values, and zero incorrectly formatted dates. Numeric inconsistencies such as negative units sold, negative sales amount, and discount values greater than 100% were corrected using median-based imputation.

In [44]:
# Phase 3: Speak One Language
text_cols = ['Customer_Name', 'Product_Category', 'Product_Name', 'City', 'Segment']

for col in text_cols:
  df[col] = df[col].astype(str).str.strip().str.title()


In [46]:
df['City'].unique()

array(['Delhi', 'Ahmedabad', 'Mumbai', 'Bengaluru', 'Hyderbad',
       'Bangalore', 'Pune', 'Hyd', 'Hyderabad'], dtype=object)

In [48]:
# 2. Standardize city names
city_mapping = {
    'Delhi': 'Delhi',
    'Ahmedabad': 'Ahmedabad',
    'Mumbai': 'Mumbai',
    'Bengaluru': 'Bengaluru',
    'Bangalore': 'Bengaluru',
    'Hyd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    'Hyderabad':'Hyderabad',
    'Pune' : 'Pune'}
df['City'] = df['City'].replace(city_mapping)



In [49]:
# 3. Standardize date format
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Order_Date'] = df['Order_Date'].dt.strftime('%Y-%m-%d')

In [50]:
# 4. Round numeric values to 2 decimals
df['Unit_Price'] = df['Unit_Price'].round(2)
df['Discount_%'] = df['Discount_%'].round(2)
df['Sales_Amount'] = df['Sales_Amount'].round(2)
df['Profit'] = df['Profit'].round(2)


In [53]:
# 5. Final check
print(df['City'].unique())

print(df['Order_Date'].head())

print(df.dtypes)

['Delhi' 'Ahmedabad' 'Mumbai' 'Bengaluru' 'Hyderabad' 'Pune']
0    2025-02-27
1    2025-07-15
2    2024-03-27
3    2024-03-27
4    2024-03-27
Name: Order_Date, dtype: object
Order_ID              int64
Customer_Name        object
Product_Category     object
Product_Name         object
Units_Sold          float64
Unit_Price          float64
Discount_%          float64
Sales_Amount        float64
Order_Date           object
City                 object
Segment              object
Profit              float64
dtype: object


In Phase 3, inconsistent formats were standardized to ensure the dataset follows one common structure. Text fields were trimmed and converted to proper case. City names were standardized to remove spelling variations such as Bangalore, BLR, and Bengaluru. Dates were converted into ISO 8601 format (YYYY-MM-DD), and numeric columns were rounded to two decimal places.

In [55]:
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Order_ID:", df['Order_ID'].duplicated().sum())
print("Invalid dates:", pd.to_datetime(df['Order_Date'], errors='coerce').isnull().sum())
print("Negative Units_Sold:", (df['Units_Sold'] < 0).sum())
print("Negative Sales_Amount:", (df['Sales_Amount'] < 0).sum())
print("Discount above 100%:", (df['Discount_%'] > 1).sum())
print("Cities:", df['City'].unique())

Missing values:
 Order_ID            0
Customer_Name       0
Product_Category    0
Product_Name        0
Units_Sold          0
Unit_Price          0
Discount_%          0
Sales_Amount        0
Order_Date          0
City                0
Segment             0
Profit              0
dtype: int64
Duplicate rows: 0
Duplicate Order_ID: 0
Invalid dates: 0
Negative Units_Sold: 0
Negative Sales_Amount: 0
Discount above 100%: 0
Cities: ['Delhi' 'Ahmedabad' 'Mumbai' 'Bengaluru' 'Hyderabad' 'Pune']


In [56]:
df.to_csv("cleaned_snitch_fashion_sales.csv", index=False)